In [1]:
%pip install -U "sagemaker<3.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.7 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.9 MB/s  0:00:00


  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0


  Attempting uninstall: attrs


    Found existing installation: attrs 26.1.0
    Uninstalling attrs-26.1.0:
      Successfully uninstalled attrs-26.1.0
   ━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━ 3/7 [attrs]

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [sagemaker]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.3.0 which is incompatible.
sagemaker-mlops 1.11.0 requires sagemaker-core>=2.11.0, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-serve 1.11.0 requires sagemaker-core>=2.11.0, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-studio-analytic

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
import time
import boto3
import sagemaker
import pandas as pd
import numpy as np

from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterInteger, ParameterString, ParameterFloat
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, TransformStep
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet, Join
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput, TransformInput
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.transformer import Transformer

sagemaker_session = sagemaker.Session()
pipeline_session = PipelineSession()

REGION = sagemaker_session.boto_region_name
role = sagemaker.get_execution_role()

account_id = boto3.client("sts").get_caller_identity()["Account"]
BUCKET_NAME = f"aai540-diabetes-readmission-{account_id}"

s3 = boto3.client("s3", region_name=REGION)

try:
    s3.head_bucket(Bucket=BUCKET_NAME)
    print("Bucket already exists:", BUCKET_NAME)
except Exception:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3.create_bucket(
            Bucket=BUCKET_NAME,
            CreateBucketConfiguration={"LocationConstraint": REGION}
        )
    print("Bucket created:", BUCKET_NAME)

RAW_FILE = "diabetic_data.csv"
TARGET = "readmitted"
RANDOM_STATE = 42
SAMPLE_SIZE = 15000
PRODUCTION_SIZE = 6000

print("Region:", REGION)
print("Role:", role)
print("Bucket:", BUCKET_NAME)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Bucket created: aai540-diabetes-readmission-468962265940
Region: us-east-1
Role: arn:aws:iam::468962265940:role/LabRole
Bucket: aai540-diabetes-readmission-468962265940


In [2]:
raw_s3_key = "raw/diabetes/diabetic_data.csv"

s3.upload_file(RAW_FILE, BUCKET_NAME, raw_s3_key)

raw_data_s3_uri = f"s3://{BUCKET_NAME}/{raw_s3_key}"

print("Raw data uploaded:")
print(raw_data_s3_uri)

Raw data uploaded:
s3://aai540-diabetes-readmission-468962265940/raw/diabetes/diabetic_data.csv


In [3]:
!mkdir -p code

In [4]:
%%writefile code/preprocessing.py

import argparse
import os
import json
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

TARGET = "readmitted"
RANDOM_STATE = 42

FEATURES = [
    "race",
    "gender",
    "age",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "max_glu_serum",
    "A1Cresult",
    "change",
    "diabetesMed",
    "insulin"
]


def clean_feature_name(col):
    col = str(col)
    col = re.sub(r"[^a-zA-Z0-9]", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")
    if not re.match(r"^[a-zA-Z0-9]", col):
        col = "f_" + col
    return col[:64]


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--sample-size", type=int, default=15000)
    parser.add_argument("--production-size", type=int, default=6000)
    args = parser.parse_args()

    base_dir = "/opt/ml/processing"

    input_path = os.path.join(base_dir, "input", "diabetic_data.csv")

    df = pd.read_csv(input_path)

    # Replace original missing-value placeholder
    df = df.replace("?", np.nan)

    # Binary target:
    # 1 = readmitted within 30 days
    # 0 = not readmitted within 30 days
    df[TARGET] = df[TARGET].replace({
        "<30": 1,
        ">30": 0,
        "NO": 0
    }).astype(int)

    # Cost-safe development sample and reserved production sample
    df_sample, df_remaining = train_test_split(
        df,
        train_size=args.sample_size,
        stratify=df[TARGET],
        random_state=RANDOM_STATE
    )

    df_production_raw, _ = train_test_split(
        df_remaining,
        train_size=args.production_size,
        stratify=df_remaining[TARGET],
        random_state=RANDOM_STATE
    )

    df_sample = df_sample.reset_index(drop=True)
    df_production_raw = df_production_raw.reset_index(drop=True)

    # -----------------------------
    # Training/development encoding
    # -----------------------------

    df_model_raw = df_sample[FEATURES + [TARGET]].copy()

    X = df_model_raw.drop(columns=[TARGET])
    y = df_model_raw[TARGET]

    X_encoded = pd.get_dummies(X, drop_first=True)
    X_encoded.columns = [clean_feature_name(c) for c in X_encoded.columns]

    feature_columns = X_encoded.columns.tolist()

    df_model = X_encoded.copy()
    df_model[TARGET] = y.values

    # Convert bool to int for SageMaker XGBoost
    for col in df_model.columns:
        if df_model[col].dtype == "bool":
            df_model[col] = df_model[col].astype(int)

    train_df, temp_df = train_test_split(
        df_model,
        test_size=0.20,
        stratify=df_model[TARGET],
        random_state=RANDOM_STATE
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df[TARGET],
        random_state=RANDOM_STATE
    )

    # SageMaker built-in XGBoost expects label as first column
    def label_first(df):
        return df[[TARGET] + [c for c in df.columns if c != TARGET]]

    train_out = label_first(train_df)
    val_out = label_first(val_df)
    test_out = label_first(test_df)

    # -----------------------------
    # Production batch encoding
    # -----------------------------

    prod_raw = df_production_raw[FEATURES + [TARGET]].copy()

    X_prod = prod_raw.drop(columns=[TARGET])
    y_prod = prod_raw[TARGET]

    X_prod_encoded = pd.get_dummies(X_prod, drop_first=True)
    X_prod_encoded.columns = [clean_feature_name(c) for c in X_prod_encoded.columns]

    X_prod_encoded = X_prod_encoded.reindex(
        columns=feature_columns,
        fill_value=0
    )

    for col in X_prod_encoded.columns:
        if X_prod_encoded[col].dtype == "bool":
            X_prod_encoded[col] = X_prod_encoded[col].astype(int)

    X_prod_encoded = X_prod_encoded.apply(pd.to_numeric, errors="coerce").fillna(0)

    production_labeled = X_prod_encoded.copy()
    production_labeled[TARGET] = y_prod.values
    production_labeled = label_first(production_labeled)

    # Batch Transform input must be feature-only, no header
    batch_features = X_prod_encoded.copy()

    # -----------------------------
    # Save outputs
    # -----------------------------

    os.makedirs(f"{base_dir}/train", exist_ok=True)
    os.makedirs(f"{base_dir}/validation", exist_ok=True)
    os.makedirs(f"{base_dir}/test", exist_ok=True)
    os.makedirs(f"{base_dir}/batch", exist_ok=True)
    os.makedirs(f"{base_dir}/monitoring", exist_ok=True)

    train_out.to_csv(f"{base_dir}/train/train.csv", header=False, index=False)
    val_out.to_csv(f"{base_dir}/validation/validation.csv", header=False, index=False)
    test_out.to_csv(f"{base_dir}/test/test.csv", header=False, index=False)

    batch_features.to_csv(f"{base_dir}/batch/batch_input.csv", header=False, index=False)
    production_labeled.to_csv(f"{base_dir}/monitoring/production_labeled.csv", header=True, index=False)

    schema = {
        "target": TARGET,
        "feature_columns": feature_columns,
        "train_rows": int(len(train_out)),
        "validation_rows": int(len(val_out)),
        "test_rows": int(len(test_out)),
        "production_rows": int(len(batch_features))
    }

    with open(f"{base_dir}/monitoring/schema.json", "w") as f:
        json.dump(schema, f, indent=2)

    print("Preprocessing complete.")
    print(schema)

Writing code/preprocessing.py


In [5]:
%%writefile code/evaluation.py

import os
import json
import tarfile
import pathlib
import argparse

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--prediction-threshold", type=float, default=0.50)
    args = parser.parse_args()

    model_tar_path = "/opt/ml/processing/model/model.tar.gz"

    with tarfile.open(model_tar_path) as tar:
        tar.extractall(path="/opt/ml/processing/model")

    model_file = "/opt/ml/processing/model/xgboost-model"

    booster = xgb.Booster()
    booster.load_model(model_file)

    test_path = "/opt/ml/processing/test/test.csv"

    df = pd.read_csv(test_path, header=None)

    y_test = df.iloc[:, 0].astype(int)
    X_test = df.iloc[:, 1:]

    dtest = xgb.DMatrix(X_test)

    probabilities = booster.predict(dtest)
    predictions = (probabilities >= args.prediction_threshold).astype(int)

    report = {
        "classification_metrics": {
            "accuracy": {
                "value": float(accuracy_score(y_test, predictions))
            },
            "precision": {
                "value": float(precision_score(y_test, predictions, zero_division=0))
            },
            "recall": {
                "value": float(recall_score(y_test, predictions, zero_division=0))
            },
            "f1": {
                "value": float(f1_score(y_test, predictions, zero_division=0))
            },
            "roc_auc": {
                "value": float(roc_auc_score(y_test, probabilities))
            },
            "confusion_matrix": {
                "value": confusion_matrix(y_test, predictions).tolist()
            },
            "prediction_threshold": {
                "value": float(args.prediction_threshold)
            }
        }
    }

    output_dir = "/opt/ml/processing/evaluation"
    pathlib.Path(output_dir).mkdir(parents=True, exist_ok=True)

    evaluation_path = f"{output_dir}/evaluation.json"

    with open(evaluation_path, "w") as f:
        json.dump(report, f, indent=2)

    print(json.dumps(report, indent=2))

Writing code/evaluation.py


In [6]:
processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount",
    default_value=1
)

processing_instance_type = ParameterString(
    name="ProcessingInstanceType",
    default_value="ml.m5.large"
)

training_instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m5.large"
)

transform_instance_type = ParameterString(
    name="TransformInstanceType",
    default_value="ml.m5.large"
)

model_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="PendingManualApproval"
)

input_data = ParameterString(
    name="InputData",
    default_value=raw_data_s3_uri
)

f1_threshold = ParameterFloat(
    name="F1Threshold",
    default_value=0.20
)

prediction_threshold = ParameterFloat(
    name="PredictionThreshold",
    default_value=0.50
)

sample_size_param = ParameterInteger(
    name="SampleSize",
    default_value=15000
)

production_size_param = ParameterInteger(
    name="ProductionSize",
    default_value=6000
)

In [7]:
sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    base_job_name="diabetes-preprocess",
    sagemaker_session=pipeline_session
)

processor_args = sklearn_processor.run(
    inputs=[
        ProcessingInput(
            source=input_data,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/validation"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test"
        ),
        ProcessingOutput(
            output_name="batch",
            source="/opt/ml/processing/batch"
        ),
        ProcessingOutput(
            output_name="monitoring",
            source="/opt/ml/processing/monitoring"
        )
    ],
    code="code/preprocessing.py",
    arguments=[
        "--sample-size", sample_size_param.to_string(),
        "--production-size", production_size_param.to_string()
    ]
)

step_process = ProcessingStep(
    name="DiabetesPreprocess",
    step_args=processor_args
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [8]:
xgboost_image_uri = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=REGION,
    version="1.7-1",
    py_version="py3",
    instance_type="ml.m5.large"
)

model_output_path = f"s3://{BUCKET_NAME}/pipeline/model-artifacts"

xgb_estimator = Estimator(
    image_uri=xgboost_image_uri,
    role=role,
    instance_count=1,
    instance_type=training_instance_type,
    output_path=model_output_path,
    sagemaker_session=pipeline_session
)

# Good simple starting point for imbalanced binary classification
xgb_estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="logloss",
    num_round=150,
    max_depth=4,
    eta=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=8
)

train_args = xgb_estimator.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv"
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv"
        )
    }
)

step_train = TrainingStep(
    name="DiabetesTrainXGBoost",
    step_args=train_args
)

INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.large.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [9]:
script_eval = ScriptProcessor(
    image_uri=xgboost_image_uri,
    command=["python3"],
    role=role,
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="diabetes-evaluate",
    sagemaker_session=pipeline_session
)

eval_args = script_eval.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model"
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation"
        )
    ],
    code="code/evaluation.py",
    arguments=[
        "--prediction-threshold", prediction_threshold.to_string()
    ]
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

step_eval = ProcessingStep(
    name="DiabetesEvaluate",
    step_args=eval_args,
    property_files=[evaluation_report]
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [10]:
model = Model(
    image_uri=xgboost_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session
)

step_create_model = ModelStep(
    name="DiabetesCreateModel",
    step_args=model.create(
        instance_type="ml.m5.large"
    )
)

transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type=transform_instance_type,
    instance_count=1,
    output_path=f"s3://{BUCKET_NAME}/pipeline/batch-transform-output",
    sagemaker_session=pipeline_session
)

step_transform = TransformStep(
    name="DiabetesBatchTransform",
    transformer=transformer,
    inputs=TransformInput(
        data=step_process.properties.ProcessingOutputConfig.Outputs["batch"].S3Output.S3Uri,
        content_type="text/csv",
        split_type="Line"
    )
)

model_package_group_name = "DiabetesReadmissionModelPackageGroup"

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json"
    )
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics
)

step_register = ModelStep(
    name="DiabetesRegisterModel",
    step_args=register_args
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [11]:
step_fail = FailStep(
    name="DiabetesF1Fail",
    error_message=Join(
        on=" ",
        values=[
            "Pipeline failed because F1 score was below threshold:",
            f1_threshold
        ]
    )
)

condition_f1 = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="classification_metrics.f1.value"
    ),
    right=f1_threshold
)

step_condition = ConditionStep(
    name="DiabetesF1Condition",
    conditions=[condition_f1],
    if_steps=[
        step_create_model,
        step_register,
        step_transform
    ],
    else_steps=[
        step_fail
    ]
)

In [12]:
pipeline_name = "DiabetesReadmissionPipeline"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        processing_instance_type,
        training_instance_type,
        transform_instance_type,
        model_approval_status,
        input_data,
        f1_threshold,
        prediction_threshold,
        sample_size_param,
        production_size_param
    ],
    steps=[
        step_process,
        step_train,
        step_eval,
        step_condition
    ],
    sagemaker_session=pipeline_session
)

definition = json.loads(pipeline.definition())

print("Pipeline definition created.")
print("Pipeline name:", pipeline_name)

Pipeline definition created.
Pipeline name: DiabetesReadmissionPipeline


In [13]:
pipeline.upsert(role_arn=role)

print("Pipeline upsert complete.")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline upsert complete.


In [14]:
success_execution = pipeline.start(
    parameters={
        "F1Threshold": 0.20,
        "PredictionThreshold": 0.50,
        "ModelApprovalStatus": "PendingManualApproval"
    }
)

print("Successful pipeline execution started:")
print(success_execution.arn)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Successful pipeline execution started:
arn:aws:sagemaker:us-east-1:468962265940:pipeline/DiabetesReadmissionPipeline/execution/0b9vji2gqr9j


In [15]:
failed_execution = pipeline.start(
    parameters={
        "F1Threshold": 0.99,
        "PredictionThreshold": 0.50,
        "ModelApprovalStatus": "PendingManualApproval"
    }
)

print("Failed pipeline execution started:")
print(failed_execution.arn)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Failed pipeline execution started:
arn:aws:sagemaker:us-east-1:468962265940:pipeline/DiabetesReadmissionPipeline/execution/31waeir72jov


In [21]:
success_execution.list_steps()

[{'StepName': 'DiabetesBatchTransform',
  'StartTime': datetime.datetime(2026, 6, 18, 20, 44, 9, 723000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 6, 18, 20, 50, 7, 682000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'TransformJob': {'Arn': 'arn:aws:sagemaker:us-east-1:468962265940:transform-job/pipelines-0b9vji2gqr9j-DiabetesBatchTransfo-e7WwaNL7R1'}},
  'AttemptCount': 1},
 {'StepName': 'DiabetesCreateModel-CreateModel',
  'StartTime': datetime.datetime(2026, 6, 18, 20, 44, 7, 967000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 6, 18, 20, 44, 9, 333000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'Model': {'Arn': 'arn:aws:sagemaker:us-east-1:468962265940:model/pipelines-0b9vji2gqr9j-DiabetesCreateModel--K9w8OOTjuI'}},
  'AttemptCount': 1},
 {'StepName': 'DiabetesRegisterModel-RegisterModel',
  'StartTime': datetime.datetime(2026, 6, 18, 20, 44, 7, 967000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 6, 18, 20,

In [22]:
from pprint import pprint
from sagemaker.s3 import S3Downloader

evaluation_s3_uri = "{}/evaluation.json".format(
    step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
)

evaluation_json = S3Downloader.read_file(evaluation_s3_uri)

evaluation_report_dict = json.loads(evaluation_json)

pprint(evaluation_report_dict)

{'classification_metrics': {'accuracy': {'value': 0.682},
                            'confusion_matrix': {'value': [[944, 389],
                                                           [88, 79]]},
                            'f1': {'value': 0.24881889763779527},
                            'precision': {'value': 0.16880341880341881},
                            'prediction_threshold': {'value': 0.5},
                            'recall': {'value': 0.47305389221556887},
                            'roc_auc': {'value': 0.6437777108947896}}}


In [23]:
from datetime import datetime, timezone

cw = boto3.client("cloudwatch", region_name=REGION)

metrics = evaluation_report_dict["classification_metrics"]

namespace = "AAI540/DiabetesReadmissionPipeline"

metric_data = []

for metric_name in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    metric_data.append({
        "MetricName": metric_name.upper(),
        "Timestamp": datetime.now(timezone.utc),
        "Value": float(metrics[metric_name]["value"]),
        "Unit": "None",
        "Dimensions": [
            {"Name": "Project", "Value": "DiabetesReadmission"},
            {"Name": "Pipeline", "Value": pipeline_name}
        ]
    })

cw.put_metric_data(
    Namespace=namespace,
    MetricData=metric_data
)

print("Published metrics to CloudWatch namespace:")
print(namespace)

Published metrics to CloudWatch namespace:
AAI540/DiabetesReadmissionPipeline


In [24]:
dashboard_name = "AAI540-DiabetesReadmission-Pipeline-Monitoring"

dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0,
            "y": 0,
            "width": 24,
            "height": 6,
            "properties": {
                "metrics": [
                    [namespace, "ACCURACY", "Project", "DiabetesReadmission", "Pipeline", pipeline_name],
                    [namespace, "PRECISION", "Project", "DiabetesReadmission", "Pipeline", pipeline_name],
                    [namespace, "RECALL", "Project", "DiabetesReadmission", "Pipeline", pipeline_name],
                    [namespace, "F1", "Project", "DiabetesReadmission", "Pipeline", pipeline_name],
                    [namespace, "ROC_AUC", "Project", "DiabetesReadmission", "Pipeline", pipeline_name]
                ],
                "view": "timeSeries",
                "stacked": False,
                "region": REGION,
                "title": "Diabetes Readmission Model Quality Metrics",
                "period": 300,
                "stat": "Average"
            }
        }
    ]
}

cw.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body)
)

print("CloudWatch dashboard created:")
print(dashboard_name)

CloudWatch dashboard created:
AAI540-DiabetesReadmission-Pipeline-Monitoring
